This notebook solves the liquid-cooling BTMS 1D S-flow topology model.

Updates in this version: the 0D reference model has been removed; the heat-generation workbook is read only from `18650FAC.xlsx` in the same directory; the plot reads only `min_avg.out` and `max_avg.out`; and the aluminum cold-plate conduction thickness is changed to `t_plate/2` because the liquid flow domain is inside the plate and the aluminum heat capacity is not modeled.


In [ ]:
# This file was sent by Liu Mingxin, who update the 'BTMS eval vs power generattion.ipynb' to account for liquid cooling. 
# The original file saved with an FID: HP02E030107A2  code_liquid_cooling_BLC_15cells_1200s_newFAC_modified_no_0D_with_natconv_Liu_20260518.ipynb

# read the newest 18650FAC heat-generation profile from the same directory as this notebook

import os
import sys
import numpy as np
import pandas as pd

# User-provided heat-generation file. Keep the normal local filename first.
# In this chat environment the newest uploaded copy is 18650FAC(2).xlsx, so it is
# checked before the older 18650FAC(1).xlsx fallback.
# CANDIDATE_DATA_FILES = ["18650FAC.xlsx", "18650FAC(2).xlsx", "18650FAC(1).xlsx"]


# update the file name for the current folder



# def find_data_file(candidate_names):
#     """Find the heat-generation workbook in the current notebook directory."""
#     search_dirs = [os.getcwd()]
#     for directory in search_dirs:
#         for name in candidate_names:
#             path = os.path.join(directory, name)
#             if os.path.exists(path):
#                 return path
#     raise FileNotFoundError(
#         "Could not find 18650FAC.xlsx in the notebook directory. "
#         "Please place the newest 18650FAC workbook next to this notebook."
#     )


def normalize_column_name(name):
    """Normalize column names for robust matching."""
    return str(name).strip().lower().replace(" ", "")


def find_column(dataframe, candidates):
    """Find a column by case/space-insensitive matching."""
    normalized = {normalize_column_name(col): col for col in dataframe.columns}
    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in normalized:
            return normalized[key]
    raise KeyError(f"None of the expected columns were found: {candidates}. Existing columns: {list(dataframe.columns)}")


def numeric_series(dataframe, candidates):
    """Return a numeric Series from the first matching candidate column."""
    col = find_column(dataframe, candidates)
    series = pd.to_numeric(dataframe[col], errors="coerce").dropna()
    return col, series


def first_float_or_none(dataframe, candidates):
    """Read the first numeric value from a candidate column, if it exists."""
    try:
        _, values = numeric_series(dataframe, candidates)
    except KeyError:
        return None
    if values.empty:
        return None
    return float(values.iloc[0])



In [ ]:
# Read the heat-generation Excel workbook ONLY from the current notebook directory.
# Put this file in the same folder as this .ipynb:
#   18650FAC.xlsx

from pathlib import Path

DATA_FILE_NAME = "18650FAC.xlsx"
file_name = Path.cwd() / DATA_FILE_NAME

if not file_name.exists():
    raise FileNotFoundError(
        f"Cannot find {DATA_FILE_NAME} in the current notebook directory:\n"
        f"  {Path.cwd()}\n\n"
        f"Please put {DATA_FILE_NAME} in the same folder as this notebook, then rerun."
    )

print(f"Reading heat-generation file: {file_name}")
df = pd.read_excel(file_name, sheet_name="Sheet1")

# Geometry values in the 18650FAC file.
H_bat_from_file = first_float_or_none(df, ["H(高度)", "H", "height", "Height"])
D_bat_from_file = first_float_or_none(df, ["D(直径)", "D", "diameter", "Diameter"])
H_bat_for_q = H_bat_from_file if H_bat_from_file is not None else 65e-3
D_bat_for_q = D_bat_from_file if D_bat_from_file is not None else 18e-3
V_bat_for_q = np.pi * D_bat_for_q ** 2 / 4.0 * H_bat_for_q

# The 18650FAC file has a complete Qbat(W) column, so use it directly as
# single-cell heat-generation power. Do not use Qpack(W), because this model
# assembles module heat from the single-cell values internally.
try:
    power_col, power_series = numeric_series(df, ["Qbat(W)", "Qbat", "qbat(W)"])
    power_generation_data = power_series.to_numpy(dtype=float)
    power_source_note = f"{power_col} directly, single-cell heat generation"
except KeyError:
    # Fallback: if the workbook omits Qbat(W), compute single-cell power from qbat(W/m^3).
    qvol_col, qvol_series = numeric_series(df, ["qbat(W/m^3)", "qbat(W/m3)", "qbat", "qgen(W/m^3)", "qgen"])
    power_generation_data = qvol_series.to_numpy(dtype=float) * V_bat_for_q
    power_col = qvol_col
    power_source_note = f"{qvol_col} multiplied by computed cell volume {V_bat_for_q:.8e} m^3"

# Each row is one 1-second heat-generation interval. 1200 rows => states from 0 to 1200 s.
dt = 1.0
heat_generation_steps = len(power_generation_data)
simulation_duration_s = heat_generation_steps * dt

if heat_generation_steps != 1200:
    print(f"WARNING: heat-generation data contains {heat_generation_steps} rows, not 1200 rows.")

print(f"Loaded heat-generation data from: {file_name}")
print(f"Power source: {power_source_note}")
print(f"Heat-generation intervals: {heat_generation_steps}; simulation end time: {simulation_duration_s:.0f} s")
print(f"Power range: min={np.min(power_generation_data):.6g} W, max={np.max(power_generation_data):.6g} W, mean={np.mean(power_generation_data):.6g} W per cell")
print(f"Battery geometry from file: H={H_bat_from_file} m, D={D_bat_from_file} m, V={V_bat_for_q:.8e} m^3")



In [ ]:
# Battery parameters for NCR18650PF cylindrical cells used in the Zhang et al. 2024 BLC case.
# The heat-generation file has 1200 rows. With dt = 1 s, the state arrays must have 1201 rows:
# t = 0, 1, 2, ..., 1200 s.

T_bat_init = 25.0  # initial battery temperature in Celsius
cp_bat = 877.24    # J/(kg·K), from the paper table for the 18650 cell
m_bat = 47e-3      # kg, mass of one 18650 cell
D_bat = D_bat_from_file if D_bat_from_file is not None else 18e-3  # m
H_bat = H_bat_from_file if H_bat_from_file is not None else 65e-3  # m

# 15-cell BLC module layout. The 1D liquid model uses one thermal segment per cell.
N_r, N_c = 3, 5
N_cells = N_r * N_c

A_battery = np.pi * D_bat * H_bat  # lateral surface area of one cylindrical cell, m^2
A_top_battery = np.pi * D_bat**2 / 4.0  # top end-face area exposed to ambient air, m^2

# Natural convection from the exposed battery surface to ambient air.
# Assumption used here: cylindrical side surface + top end face are exposed to air;
# the bottom end face is not included to avoid double-counting the liquid-cooling contact side.
# Positive Q_nat_conv means heat leaves the battery cell to the ambient.
# The value h = 10 W/(m^2·K) represents a stronger natural-convection assumption;
# adjust it if you have measured/CFD-calibrated data.
h_nat_conv_side = 10.0       # W/(m^2·K)
T_amb_nat_conv = 25.0        # degC
A_nat_conv_cell = A_battery + A_top_battery
A_side_nat_conv_cell = A_nat_conv_cell  # backward-compatible name used by the solver
num_time_steps = heat_generation_steps + 1

# Adiabatic single-cell temperature rise from the new heat-generation profile.
T_bat_wo_cooling = np.zeros(num_time_steps)
T_bat_wo_cooling[0] = T_bat_init

for i in range(1, num_time_steps):
    # power_generation_data[i-1] is the heat generated during interval (i-1, i].
    dT = (power_generation_data[i - 1] * dt) / (m_bat * cp_bat)
    T_bat_wo_cooling[i] = T_bat_wo_cooling[i - 1] + dT

print(
    f"Battery setup: N_cells={N_cells}, m_bat={m_bat:.6g} kg, cp_bat={cp_bat:.6g} J/(kg·K), "
    f"D_bat={D_bat:.6g} m, H_bat={H_bat:.6g} m"
)
print(f"Simulation time grid: {num_time_steps} state points, t = 0 to {(num_time_steps-1)*dt:.0f} s")
print(
    f"Natural convection: h={h_nat_conv_side:.6g} W/(m^2·K), "
    f"T_amb={T_amb_nat_conv:.6g} °C, A_nat={A_nat_conv_cell:.6g} m^2 per cell "
    f"(side + top, bottom excluded)"
)


In [ ]:
# Initial conditions and BLC liquid-cooling parameters
# BLC replication case from section 3.1: Tamb = 25 °C, Tin = 25 °C, Fc = 240 mL/min.
# Only bottom liquid cooling is retained; heat-pipe material data are intentionally not used.

import numpy as np

# Liquid coolant settings
T_liq_in = 25.0  # °C
T_liq_in_K = T_liq_in + 273.15
p_liq = 101325  # Pa
fluid_liq = "Water"

# Coolant properties from the paper table
rho_liq_const = 998.2   # kg/m^3
cp_liq_const = 4182.0   # J/(kg·K)
k_liq_const = 0.6       # W/(m·K)

# Paper flow rate Fc = 240 mL/min. Convert volumetric flow rate to mass flow rate.
Fc_mL_min = 240.0
vol_flow_m3_s = Fc_mL_min * 1e-6 / 60.0  # 1 mL = 1e-6 m^3
m_dot_liq = rho_liq_const * vol_flow_m3_s # kg/s

# Cold-plate / liquid-channel geometry.
D_channel = 4e-3      # m, circular channel diameter / hydraulic diameter
D_h = D_channel       # m, hydraulic diameter for circular channel, same as diameter

# S-shaped channel: 4 physical liquid columns × 5 vertical positions = 20 liquid segments.
# The liquid unknowns are stored in S-flow order, i.e. fluid 1 -> fluid 2 -> ... -> fluid 20.
L_channel_total = 144e-3 * 4
N_cool_seg = 20

# -----------------------------------------------------------------------------
# Battery-fluid topology for the 3-column × 5-cell layout.
# Python uses 0-based indices below; comments show the user's 1-based numbering.
#
# Battery cells:
#   column 1: b1,  b2,  b3,  b4,  b5
#   column 2: b6,  b7,  b8,  b9,  b10
#   column 3: b11, b12, b13, b14, b15
#
# Liquid segments are already ordered by the S-shaped flow path:
#   f1 -> f2 -> ... -> f20
#
# Physical adjacency used for heat transfer:
#   b1  <-> f1
#   b2  <-> f2
#   b3  <-> f3
#   b4  <-> f4
#   b5  <-> f5
#   b10 <-> f6  and f15
#   b9  <-> f7  and f14
#   b8  <-> f8  and f13
#   b7  <-> f9  and f12
#   b6  <-> f10 and f11
#   b11 <-> f20
#   b12 <-> f19
#   b13 <-> f18
#   b14 <-> f17
#   b15 <-> f16
#
# Note: the last mapping is f16, not f26, because this model has 20 liquid segments.
# -----------------------------------------------------------------------------
cell_to_cool_map_1based = {
    1: (1,),
    2: (2,),
    3: (3,),
    4: (4,),
    5: (5,),
    6: (10, 11),
    7: (9, 12),
    8: (8, 13),
    9: (7, 14),
    10: (6, 15),
    11: (20,),
    12: (19,),
    13: (18,),
    14: (17,),
    15: (16,),
}

# Convert to 0-based indices for array access.
cell_to_cool_map = {
    cell_id - 1: tuple(seg_id - 1 for seg_id in seg_ids)
    for cell_id, seg_ids in cell_to_cool_map_1based.items()
}

# Reverse map: for each fluid segment, which battery cell(s) exchange heat with it.
cool_to_cell_map = {seg: [] for seg in range(N_cool_seg)}
for cell, segs in cell_to_cool_map.items():
    for seg in segs:
        cool_to_cell_map[seg].append(cell)

# Sanity checks to catch indexing mistakes early.
assert set(cell_to_cool_map.keys()) == set(range(N_cells)), "cell_to_cool_map must contain all 15 cells."
assert all(0 <= seg < N_cool_seg for segs in cell_to_cool_map.values() for seg in segs), "Fluid segment index out of range."
assert all(len(cells) >= 1 for cells in cool_to_cell_map.values()), "Every fluid segment should touch at least one battery cell."

print("Battery -> S-flow liquid segment topology, 1-based numbering:")
for cell_id in range(1, N_cells + 1):
    print(f"  battery {cell_id:2d} -> fluid {cell_to_cool_map_1based[cell_id]}")

A_channel_cross = np.pi * D_channel ** 2 / 4.0
P_channel_wet = np.pi * D_channel

# Heat-transfer area for the internal circular liquid domain.
# The liquid domain is a circular channel carved inside the aluminum plate.
# Use the full circular-channel wall area: A = pi * D_channel * L.
#
# Correction requested for the real geometry:
# segment 1 and segment 20 are longer because the channel extends outside the
# nominal battery-covered region.  Their lengths are set to 2 times the normal
# segment length.  Other segments keep the normal length.
L_cool_seg_base = L_channel_total / N_cool_seg
L_cool_seg = np.ones(N_cool_seg, dtype=float) * L_cool_seg_base
L_cool_seg[0] = 2.0 * L_cool_seg_base
L_cool_seg[-1] = 2.0 * L_cool_seg_base
L_channel_total_effective = float(np.sum(L_cool_seg))

# Segment-wise full circular side area. Shape: (20,).
A_HT_seg = P_channel_wet * L_cool_seg
A_HT_all = float(np.sum(A_HT_seg))
A_liq_total = A_HT_all

W_plate = 76e-3 # m, cold plate width, same as the paper; retained only for reference
L_plate = 138e-3 # m, cold plate length, same as the paper; retained only for reference
A_plate_total = W_plate * L_plate # m^2, cold-plate projected area, not used as heat-transfer area


In [ ]:
# Liquid-cooling Nusselt correlation
# 删除原空气用 Žukauskas 函数和列修正因子，改为管内水冷用 Nu 计算。

def liquid_nusselt_number(Re, Pr, heating=True):
    """
    Nusselt number for internal liquid flow in a circular channel.

    Model used here:
    - Laminar fully-developed constant-wall-temperature pipe flow: Nu = 3.66 for Re < 2300
    - Turbulent Dittus-Boelter: Nu = 0.023 Re^0.8 Pr^n for Re >= 4000
      n = 0.4 when the coolant is heated by the wall/battery; n = 0.3 when the coolant is cooled.
    - Transitional region 2300 <= Re < 4000: linear interpolation between the two values.
    """
    Re = float(Re)
    Pr = float(Pr)

    if Re <= 0 or Pr <= 0:
        raise ValueError("Re and Pr must be positive for the liquid Nusselt calculation.")

    Nu_laminar = 4.36
    n = 0.4 if heating else 0.3
    
    # # use DB correlation 
    # Nu_DB = 0.023 * (Re ** 0.8) * (Pr ** n)
    
    # return Nu_DB

    def Nu_DB(Re_value):
        return 0.023 * (Re_value ** 0.8) * (Pr ** n)

    if Re < 2300:
        return Nu_laminar
    if Re >= 4000:
        return Nu_DB(Re)

    # Linear interpolation in the transition region.
    Nu_2300 = Nu_laminar
    Nu_4000 = Nu_DB(4000.0)
    weight = (Re - 2300.0) / (4000.0 - 2300.0)
    return Nu_2300 + weight * (Nu_4000 - Nu_2300)


In [ ]:
# Liquid 1D residual and solver
# 不考虑冷板比热容，也不引入冷板储能项。
# 每一段只做稳态液体能量平衡：
# Q_to_liq = h*A*(T_bat - T_liq_bulk)
# Q_from_cool = m_dot_liq * cp_liq * (T_liq_out - T_liq_in_seg)
#
# 重要更新：
#   1) 液体温度变量仍按 S 型流动顺序 f1 -> f2 -> ... -> f20 存储和推进；
#   2) 电池-流体换热使用 cell_to_cool_map 给出的物理邻接关系；
#   3) 冷却液能量方程按“每一个流体 segment”分别列残差，不再把两个流体段简单合并成一个电池残差；
#   4) 电池能量方程加入自然对流散热：
#      Q_nat = h_nat * A_nat * (T_bat - T_amb) * dt。
#      这个项只从电池节点带走热量，不进入冷却液能量方程。

import scipy.optimize as opt
import copy

def cal_BTMS_heat_balance(T_dist, args, return_segment=False):
    """Calculate heat balance for the S-flow liquid topology.

    Parameters
    ----------
    T_dist : array-like
        [T_liq_out_1, ..., T_liq_out_20, T_bat_1, ..., T_bat_15]
        The first 20 liquid temperatures are stored in S-flow order.
    args : dict
        Must contain cell_to_cool_map. Optional cool_to_cell_map is not required here.
        Natural-convection arguments are optional; if omitted, natural convection is 0.
    return_segment : bool
        If True, also return per-fluid-segment heat-transfer and enthalpy-change arrays.

    Returns
    -------
    Q_cool_HT_cell : ndarray, shape (num_seg_bat,)
        Heat removed from each battery cell by the liquid during this time step, J.
    Q_cool_change_cell : ndarray, shape (num_seg_bat,)
        Sum of coolant enthalpy changes in the segment(s) adjacent to each cell, J.
        This is mainly for diagnostics and plotting.
    Q_bat : ndarray, shape (num_seg_bat,)
        Battery internal-energy increase during this time step, J.
    Q_nat_conv_cell : ndarray, shape (num_seg_bat,)
        Natural-convection heat from each battery cell to ambient air, J.
        Positive means heat leaves the battery; negative means ambient heats the battery.
    Q_cool_HT_seg, Q_cool_change_seg : ndarray, optional, shape (num_seg_cool,)
        Per-liquid-segment heat received from the adjacent cell(s), and per-segment
        coolant enthalpy increase, both in J.
    """

    num_seg_bat = args.get("num_seg_bat")
    num_seg_cool = len(T_dist) - num_seg_bat

    dt = args.get("dt")
    T_bat_pre = args.get("T_bat_pre")
    u_cool = args.get("u_cool")
    A_HT_seg = np.asarray(args.get("A_HT_seg"), dtype=float)
    A_cool_cs = args.get("A_cool_cs")
    T_cool_in = args.get("T_cool_in")
    htc_cool = args.get("htc_cool")
    cp_cool = args.get("cp_cool")
    rho_cool = args.get("rho_cool")
    cell_to_cool_map = args.get("cell_to_cool_map")
    debug = args.get("debug", False)

    # Natural convection from exposed battery surface to ambient air.
    # Defaults make this term vanish if the caller does not provide these arguments.
    # A_nat_conv_cell is preferred; A_side_nat_conv_cell is kept for backward compatibility.
    h_nat_conv_side = args.get("h_nat_conv_side", 0.0)
    T_amb_nat_conv = args.get("T_amb_nat_conv", 0.0)
    A_nat_conv_cell = args.get("A_nat_conv_cell", args.get("A_side_nat_conv_cell", 0.0))

    T_cool = np.asarray(T_dist[0:num_seg_cool], dtype=float)   # liquid outlet temperature of each S-flow segment
    T_bat = np.asarray(T_dist[num_seg_cool:], dtype=float)     # battery cell temperature

    Q_cool_HT_cell = np.zeros(num_seg_bat)      # J, heat transferred from each battery cell to liquid
    Q_cool_change_cell = np.zeros(num_seg_bat)  # J, diagnostic sum of adjacent segment enthalpy changes
    Q_bat = np.zeros(num_seg_bat)               # J, battery sensible energy increase
    Q_nat_conv_cell = np.zeros(num_seg_bat)     # J, natural convection from each battery cell to ambient

    Q_cool_HT_seg = np.zeros(num_seg_cool)      # J, heat received by each liquid segment from adjacent battery cell(s)
    Q_cool_change_seg = np.zeros(num_seg_cool)  # J, enthalpy rise of each liquid segment

    # 1) Battery -> liquid heat transfer using physical adjacency.
    for cell in range(num_seg_bat):
        for seg in cell_to_cool_map[cell]:
            # S-flow inlet/outlet relation: segment 1 inlet is global inlet;
            # segment k inlet is segment k-1 outlet in S-flow order.
            T_cool_seg_in = T_cool_in if seg == 0 else T_cool[seg - 1]
            T_cool_seg_out = T_cool[seg]
            T_cool_bar = 0.5 * (T_cool_seg_in + T_cool_seg_out)

            # Signed heat transfer. If the coolant locally becomes warmer than the cell,
            # this term naturally becomes negative instead of being artificially clipped.
            Q_seg = htc_cool * A_HT_seg[seg] * (T_bat[cell] - T_cool_bar) * dt

            Q_cool_HT_cell[cell] += Q_seg
            Q_cool_HT_seg[seg] += Q_seg

    # 2) Liquid enthalpy increase for each S-flow segment.
    for seg in range(num_seg_cool):
        T_cool_seg_in = T_cool_in if seg == 0 else T_cool[seg - 1]
        T_cool_seg_out = T_cool[seg]
        Q_cool_change_seg[seg] = u_cool * A_cool_cs * rho_cool * cp_cool * (T_cool_seg_out - T_cool_seg_in) * dt

    # 3) Battery sensible-energy increase, natural convection, and diagnostic cell-side coolant enthalpy sum.
    for cell in range(num_seg_bat):
        dT_bat = T_bat[cell] - T_bat_pre[cell]
        Q_bat[cell] = m_bat * cp_bat * dT_bat

        # Natural convection energy term for the battery energy equation:
        # Q_nat_conv = h_nat * A_nat * (T_cell - T_amb) * dt.
        # Positive value means heat leaves the battery cell to ambient air.
        Q_nat_conv_cell[cell] = h_nat_conv_side * A_nat_conv_cell * (T_bat[cell] - T_amb_nat_conv) * dt

        Q_cool_change_cell[cell] = sum(Q_cool_change_seg[seg] for seg in cell_to_cool_map[cell])

    if debug:
        print("Per-cell heat removed by liquid [J]:", Q_cool_HT_cell)
        print("Per-cell natural-convection heat [J]:", Q_nat_conv_cell)
        print("Per-segment heat received by liquid [J]:", Q_cool_HT_seg)
        print("Per-segment liquid enthalpy rise [J]:", Q_cool_change_seg)

    if return_segment:
        return Q_cool_HT_cell, Q_cool_change_cell, Q_bat, Q_nat_conv_cell, Q_cool_HT_seg, Q_cool_change_seg
    return Q_cool_HT_cell, Q_cool_change_cell, Q_bat, Q_nat_conv_cell


def cal_liquid_power_residual(T_dist, args):
    """Residual vector for least_squares.

    The residual contains:
      - 20 liquid-segment energy balances:
        Q_received_by_liquid_segment - liquid_enthalpy_rise_segment = 0
      - 15 battery-cell energy balances:
        Q_gen - battery_sensible_increase - heat_removed_to_liquid - heat_removed_by_natconv = 0

    Word linear form for each battery cell:
        Q_gen*dt - m_bat*cp_bat*(T_bat - T_bat_pre)
        - Q_liq - h_nat*A_nat*(T_bat - T_amb)*dt = 0
    """

    num_seg_bat = args.get("num_seg_bat")
    Q_gen_battery = np.ones(num_seg_bat) * args.get("Q_gen") * args.get("dt")

    Q_cool_HT_cell, Q_cool_change_cell, Q_bat, Q_nat_conv_cell, Q_cool_HT_seg, Q_cool_change_seg = cal_BTMS_heat_balance(
        T_dist,
        args,
        return_segment=True,
    )

    res_cool = Q_cool_HT_seg - Q_cool_change_seg
    res_bat = Q_gen_battery - Q_bat - Q_cool_HT_cell - Q_nat_conv_cell

    return np.concatenate((res_cool, res_bat))


def solve_liquid_temperature_distribution(
    args,
    tol=1e-6,
    maxiter=1000,
    debug=False,
):
    """Solve the active liquid-cooling temperature distribution along the S-shaped flow path."""

    T_cool_pre = args.get("T_cool_pre")
    T_bat_pre = args.get("T_bat_pre")
    num_seg_bat = len(T_bat_pre)
    num_seg_cool = len(T_cool_pre)
    num_var = num_seg_bat + num_seg_cool
    T_cool_in = args.get("T_cool_in")

    # 全局液冷始终开启：每个时间步都求解液体温度分布。
    # 用上一时间步液温分布作为初始猜测；第一次计算时就是入口温度分布。
    T_cool_guess = copy.deepcopy(T_cool_pre)
    T_bat_guess = copy.deepcopy(T_bat_pre)

    # 保证初始猜测在边界内。
    lower_bound = np.ones(num_var) * (T_cool_in - 20.0)
    upper_limit = max(float(np.max(T_bat_guess)) + 50.0, T_cool_in + 1.0)
    upper_bound = np.ones(num_var) * upper_limit

    T_dist = np.concatenate((T_cool_guess, T_bat_guess))

    res = opt.least_squares(
        cal_liquid_power_residual,
        T_dist,
        args=(args,),
        method="trf",
        verbose=2 if debug else 0,
        ftol=tol,
        xtol=tol,
        gtol=tol,
        max_nfev=maxiter,
        bounds=(lower_bound, upper_bound),
    )

    if debug:
        print("Optimization success:", res.success)
        print("Final residual norm:", np.linalg.norm(res.fun))
        print("Optimized liquid temperatures:", res.x[:num_seg_cool])
        print("Optimized battery temperatures:", res.x[num_seg_cool:])

    return res.x


In [ ]:

# Dynamic viscosity is required for Re/Pr/Nu but is not listed in the paper table.
# Use CoolProp for viscosity only when available; otherwise use water viscosity near 25 °C.
try:
    import CoolProp.CoolProp as CP
    mu_liq_const = CP.PropsSI("V", "T", T_liq_in_K, "P", p_liq, fluid_liq)
    prop_source = "paper rho/cp/k; CoolProp viscosity"
except Exception as exc:
    mu_liq_const = 8.90e-4  # Pa·s, approximate water viscosity at 25 °C
    prop_source = f"paper rho/cp/k; fallback viscosity because CoolProp is unavailable: {exc}"

# Liquid-side Reynolds, Nusselt, h and resistances. Nu correlation remains unchanged.
Pr_liq = cp_liq_const * mu_liq_const / k_liq_const
Re_liq = m_dot_liq * D_h / (mu_liq_const * A_channel_cross)
Nu_liq = liquid_nusselt_number(Re_liq, Pr_liq, heating=False)
htc_liq = Nu_liq * k_liq_const / D_h

# Cold-plate conduction resistance.
# The liquid domain is carved inside the aluminum cold plate, and this reduced-order
# model does NOT include aluminum thermal capacity/storage. Therefore the aluminum
# conduction path should be the distance from the battery-side plate surface to the
# internal liquid-domain wall, not the full plate thickness. If the internal channel
# is located near the mid-plane of an 8 mm plate, use half thickness as the effective
# conduction thickness.
k_plate = 202.4         # W/(m·K), aluminum cold plate conductivity from the paper
t_plate = 8e-3          # m, total cold plate thickness from the paper
channel_depth_ratio = 0.5
# Effective conduction thickness from the battery-side surface to the internal liquid wall.
t_eff_plate = channel_depth_ratio * t_plate

# Unit-area thermal resistances, m²·K/W.
R_plate = t_eff_plate / k_plate
R_conv_liq = 1.0 / htc_liq
R_total = R_plate + R_conv_liq

# Effective overall heat-transfer coefficient from battery-side plate surface to liquid bulk.
htc_global = 1.0 / R_total

debug = True  # set to True to print detailed parameters for verification

# print the above parameters for verification
print(f"Liquid coolant properties: {prop_source}")
print(f"Liquid inlet temperature: {T_liq_in:.2f} °C")
print(f"Mass flow rate: {m_dot_liq:.6g} kg/s")
print(f"Reynolds number: {Re_liq:.6g}")
print(f"Nusselt number: {Nu_liq:.6g}")
print(f"Convective heat transfer coefficient: h_liq = {htc_liq:.6g} W/(m²·K)")
print(f"Cold plate total thickness: t_plate = {t_plate:.6g} m")
print(f"Effective conduction thickness: t_eff_plate = {t_eff_plate:.6g} m = {channel_depth_ratio:g} * t_plate")
print(f"Aluminum unit-area conduction resistance: R_plate = {R_plate:.6g} m²·K/W")
print(f"Liquid convection unit-area resistance: R_conv_liq = {R_conv_liq:.6g} m²·K/W")
print(f"Total unit-area resistance: R_total = {R_total:.6g} m²·K/W")
print(f"Effective global heat-transfer coefficient: htc_global = {htc_global:.6g} W/(m²·K)")
print(f"Nominal channel length before end correction: {L_channel_total:.6g} m")
print(f"Effective channel length after segment-1/20 correction: {L_channel_total_effective:.6g} m")
print(f"Total wetted perimeter: {P_channel_wet:.6g} m")
print(f"Total wetted full-area used by the model: A_HT_all = {A_HT_all:.6g} m²")
print(f"Cold-plate projected area retained only for reference: A_plate_total = {A_plate_total:.6g} m²")
print(f"Normal segment length: {L_cool_seg_base:.6g} m")
print(f"Segment lengths: first = {L_cool_seg[0]:.6g} m, middle = {L_cool_seg[1]:.6g} m, last = {L_cool_seg[-1]:.6g} m")
print(f"Heat-transfer area: first = {A_HT_seg[0]:.6g} m², middle = {A_HT_seg[1]:.6g} m², last = {A_HT_seg[-1]:.6g} m²")


In [ ]:

t = 0 # initialize time variable
T_bat_history = np.empty(num_time_steps, dtype=object) # to store battery temperature history
T_cool_history = np.empty(num_time_steps, dtype=object) # to store coolant temperature history

Q_bat_history = np.empty(num_time_steps) # to store battery sensible-energy increase history
Q_cool_history = np.empty(num_time_steps) # to store coolant heat absorption history
Q_nat_conv_history = np.empty(num_time_steps) # to store total natural-convection heat history
Q_res_history = np.empty(num_time_steps) # to store residual history for debugging
Q_cool_history_detailed = np.empty(num_time_steps, dtype=object) # to store detailed coolant heat transfer history for each cell
Q_nat_conv_history_detailed = np.empty(num_time_steps, dtype=object) # to store detailed natural-convection heat history for each cell


for i in range(num_time_steps):
    T_bat_history[i] = np.ones(N_cells) * T_bat_init # initialize battery temperature history for this time step
    T_cool_history[i] = np.ones(N_cool_seg) * T_liq_in # initialize coolant temperature history for this time step
    Q_bat_history[i] = 0 # initialize battery sensible-energy increase history for this time step
    Q_cool_history[i] = 0 # initialize coolant heat absorption history for this time step
    Q_nat_conv_history[i] = 0 # initialize natural-convection heat history for this time step
    Q_res_history[i] = 0 # initialize residual history for this time step
    Q_cool_history_detailed[i] = np.zeros(N_cells) # initialize detailed coolant heat transfer history for this time step
    Q_nat_conv_history_detailed[i] = np.zeros(N_cells) # initialize detailed natural-convection heat history for this time step

for i in range(1, num_time_steps):

    Q_cool_HT = 0 # initialize cooling power for this time step

    print(f"*** Time step {i}: t = {t:.0f} s ***")   

    T_bat_pre = T_bat_history[i - 1]  # battery temperature from the previous time step     
    T_cool_pre = T_cool_history[i - 1]  # coolant temperature from the previous time step

    args = {
        "dt": dt,
        "num_seg_bat": N_cells,
        "num_seg_cool": N_cool_seg,
        "T_bat_pre": T_bat_pre,
        "T_cool_pre": T_cool_pre,
        "u_cool": m_dot_liq / (rho_liq_const * A_channel_cross),
        "p_cool": p_liq,
        "fluid_cool": fluid_liq,
        "A_HT_seg": A_HT_seg,
        "A_cool_cs": A_channel_cross,
        "D_bat": D_bat,
        "T_cool_in": T_liq_in,
        "is_cool": True,  # BLC case: cooling is always on
        "htc_cool": htc_global,  # effective coefficient including aluminum conduction + liquid convection
        "cp_cool": cp_liq_const,  # use constant specific heat capacity for simplicity; adjust as needed
        "rho_cool": rho_liq_const,  # use constant density for simplicity; adjust as needed
        "Q_gen": power_generation_data[i - 1],  # heat generated in the battery during this time step
        "cell_to_cool_map": cell_to_cool_map,
        "cool_to_cell_map": cool_to_cell_map,
        "h_nat_conv_side": h_nat_conv_side,
        "T_amb_nat_conv": T_amb_nat_conv,
        "A_nat_conv_cell": A_nat_conv_cell,
        "A_side_nat_conv_cell": A_side_nat_conv_cell,  # backward-compatible alias
    }

    T_dist = solve_liquid_temperature_distribution(
        args,
        tol = 1e-3,
        maxiter = 1000,
        debug = debug
    )

    Q_cool_HT, Q_cool_change, Q_bat, Q_nat_conv = cal_BTMS_heat_balance(T_dist, args)

    T_cool = T_dist[0:N_cool_seg]
    T_bat = T_dist[N_cool_seg:]    
    Q_bat_history[i] = sum(Q_bat)
    Q_cool_history[i] = sum(Q_cool_HT)
    Q_nat_conv_history[i] = sum(Q_nat_conv)
    Q_cool_history_detailed[i] = Q_cool_HT
    Q_nat_conv_history_detailed[i] = Q_nat_conv

    # identify the segment with minimum heat transfer for debugging; this segment may indicate a bottleneck in the cooling performance
    min_Q_cool_HT_seg = np.argmin(Q_cool_HT)
    print(f"Minimum liquid heat-transfer cell: battery {min_Q_cool_HT_seg + 1}, Q_cool_HT = {Q_cool_HT[min_Q_cool_HT_seg]:.6g} J")
    print(f"Total natural-convection heat: Q_nat_conv_total = {sum(Q_nat_conv):.6g} J")

    # Battery-module energy residual:
    # sum(Q_gen*dt) - sum(Q_bat) - sum(Q_liquid) - sum(Q_nat_conv)
    Q_res = abs(
        power_generation_data[i - 1] * N_cells
        - sum(Q_bat)
        - sum(Q_cool_HT)
        - sum(Q_nat_conv)
    )
    Q_res_history[i] = Q_res

    T_bat_history[i] = T_bat
    T_cool_history[i] = T_cool
    t = t + dt # update time variable


In [ ]:
# 读取同目录下的 CFD .out 文件，并与液冷 1D 结果画在同一张图上
# 只读取这两个本地文件：
#   min_avg.out
#   max_avg.out
# 请把它们和本 notebook 放在同一个文件夹。

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CFD_DATA_DIR = Path.cwd()
print(f"CFD 数据读取目录: {CFD_DATA_DIR}")


def read_out_temperature(filename, series_name, data_dir=CFD_DATA_DIR):
    """
    读取同目录中的 .out 文件。

    文件应至少包含两列数值：时间/步数 和 温度值。
    支持空格、Tab、逗号分隔；自动跳过表头和注释行。

    如果第二列温度看起来是 K（均值 > 200），自动转换为 °C；
    如果看起来已经是 °C，则保持不变。
    """
    path = Path(data_dir) / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Cannot find {filename} in current notebook directory:\n"
            f"  {Path(data_dir)}\n"
            f"Please put {filename} in the same folder as this notebook."
        )

    rows = []
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(("#", "//")):
                continue

            nums = re.findall(r"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?", line)
            if len(nums) >= 2:
                rows.append([float(nums[0]), float(nums[1])])

    if not rows:
        raise ValueError(f"{path} 中没有读到有效的两列数值数据：time 和 temperature。")

    data = pd.DataFrame(rows, columns=["time_s", "temperature"])

    # Fluent/CFD 输出常见是 K；如果数值明显大于室温摄氏度范围，则转为 °C。
    if data["temperature"].mean() > 200.0:
        data["temperature_C"] = data["temperature"] - 273.15
        unit_note = "K -> °C"
    else:
        data["temperature_C"] = data["temperature"]
        unit_note = "°C"

    print(f"Loaded {series_name}: {path}, rows={len(data)}, unit={unit_note}")
    return data[["time_s", "temperature_C"]]


In [ ]:
# 只读取用户给定的两个本地 CFD 平均温度文件。
CFD_SERIES_FILES = {
    "CFD_min": "min.out",
    "CFD_max": "max.out",
}

cfd_series = []
for label, filename in CFD_SERIES_FILES.items():
    data = read_out_temperature(filename, label)
    cfd_series.append((label, data))

# 额外给定数据：Zhang BLC-max，原始单位为 K，绘图时转换为 °C。
BLC_max = np.array([
    [0, 298.16],
    [31.30, 299.30],
    [70.31, 300.34],
    [115.61, 301.12],
    [160.61, 301.74],
    [197.32, 302.22],
    [238.62, 302.60],
    [277.63, 302.88],
    [323.52, 303.12],
    [360.23, 303.27],
    [399.24, 303.39],
    [435.95, 303.47],
    [479.54, 303.50],
    [520.84, 303.51],
    [562.14, 303.52],
    [598.85, 303.53],
    [640.15, 303.54],
    [676.86, 303.55],
    [718.16, 303.56],
    [759.46, 303.58],
    [798.47, 303.63],
    [839.77, 303.72],
    [874.19, 303.86],
    [917.78, 304.08],
    [959.08, 304.39],
    [1002.68, 304.83],
    [1039.39, 305.43],
    [1076.10, 306.18],
    [1119.69, 307.10],
    [1158.70, 308.22],
    [1197.71, 309.47],
], dtype=float)

# 额外给定数据：Zhang BLC-min，原始单位为 K，绘图时转换为 °C。
BLC_min = np.array([
    [0, 298.16],
    [39.76, 298.42],
    [81.92, 298.62],
    [121.98, 298.76],
    [157.81, 298.86],
    [197.87, 298.93],
    [240.04, 299.00],
    [286.44, 299.05],
    [320.18, 299.07],
    [358.14, 299.09],
    [402.42, 299.11],
    [438.28, 299.13],
    [478.34, 299.14],
    [520.52, 299.14],
    [558.49, 299.15],
    [604.89, 299.16],
    [649.17, 299.17],
    [678.70, 299.17],
    [720.87, 299.19],
    [763.05, 299.20],
    [798.91, 299.22],
    [843.19, 299.26],
    [883.26, 299.30],
    [923.32, 299.35],
    [961.27, 299.42],
    [995.01, 299.50],
    [1041.39, 299.63],
    [1079.34, 299.78],
    [1121.50, 299.99],
    [1161.54, 300.24],
    [1200, 300.56],
], dtype=float)


In [ ]:
# 绘制 1D 液冷结果 + 本地 min_avg/max_avg CFD 数据。
plt.figure(figsize=(12, 7))

Tmax_1D = np.array([np.max(T_bat_history[i]) for i in range(num_time_steps)])
Tmin_1D = np.array([np.min(T_bat_history[i]) for i in range(num_time_steps)])
time_s = np.arange(num_time_steps) * dt

plt.plot(time_s, Tmax_1D, linestyle="--", label="1D-max")
plt.plot(time_s, Tmin_1D, linestyle="--", label="1D-min")

# 画同目录读取的两个文件：min_avg.out 和 max_avg.out。
for label, data in cfd_series:
    plt.plot(data["time_s"], data["temperature_C"], label=label)

# 保留 Zhang BLC 曲线作为文献对照。
plt.plot(BLC_max[:, 0], BLC_max[:, 1] - 273.15, linestyle="-.", label="Zhang, BLC-max")
plt.plot(BLC_min[:, 0], BLC_min[:, 1] - 273.15, linestyle="-.", label="Zhang, BLC-min")

plt.xlabel("Simulation time (s)")
plt.ylabel("Temperature (°C)")
plt.title("Liquid Cooling 1D vs CFD min_avg/max_avg Comparison")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print("Loaded CFD series:", ", ".join([f"{label} rows={len(data)}" for label, data in cfd_series]))
